# 01 - Exploratory Data Analysis

**Goal:** understand the credit card transaction data before modelling anything, with a business lens:
*how rare is fraud, how much money is at stake, and which signals separate fraud from normal behaviour?*

Dataset: [Credit Card Fraud Detection](https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud) (ULB Machine Learning Group) -
European card transactions over two days. Features `V1`-`V28` are PCA components (anonymised), plus `Time`, `Amount` and the label `Class` (1 = fraud).

> Run `python src/download_data.py` first so `data/raw/creditcard.csv` exists.

In [ ]:
import sys
from pathlib import Path

# make `src` importable whether the notebook runs from /notebooks or the repo root
ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.config import RAW_PATH, FIG_DIR, TARGET

sns.set_style("whitegrid")
FIG_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(RAW_PATH)
df.shape

## 1. First look

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
print("Missing values:", int(df.isnull().sum().sum()))
print("Exact duplicate rows:", int(df.duplicated().sum()))

No missing values, but there are duplicate rows - they get removed in `02_cleaning.ipynb`
(duplicates can leak between train and test and inflate scores).

## 2. Class imbalance - the core challenge

In [ ]:
counts = df[TARGET].value_counts()
pct = df[TARGET].value_counts(normalize=True) * 100
pd.DataFrame({"count": counts, "percent": pct.round(3)})

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
sns.countplot(x=TARGET, data=df, ax=ax)
ax.set_yscale("log")
ax.set_title("Class distribution (0 = legitimate, 1 = fraud) - log scale")
for p in ax.patches:
    ax.annotate(f"{int(p.get_height()):,}", (p.get_x() + p.get_width() / 2, p.get_height()),
                ha="center", va="bottom")
plt.savefig(FIG_DIR / "01_class_imbalance.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Why accuracy is a trap: a "model" that never flags anything
always_legit_accuracy = (df[TARGET] == 0).mean()
print(f"Accuracy of predicting 'legitimate' every time: {always_legit_accuracy:.4%}")
print(f"Frauds caught: 0 of {int(df[TARGET].sum())}")

**Takeaway:** fewer than 1 in 500 transactions is fraud, so a model that does nothing is >99.8% "accurate" while catching zero fraud.
Accuracy is useless here - I evaluate with **precision-recall AUC** and, more importantly, with **money** (a cost function, see `src/evaluate.py`).

## 3. Money: how much is at stake?

In [ ]:
summary = df.groupby(TARGET)["Amount"].agg(["count", "mean", "median", "max", "sum"]).round(2)
summary.index = ["legitimate", "fraud"]
summary

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(data=df, x="Amount", hue=TARGET, bins=60, log_scale=(True, True),
             stat="count", common_norm=False, ax=axes[0])
axes[0].set_title("Transaction amount (log-log)")
sns.boxplot(data=df, x=TARGET, y="Amount", showfliers=False, ax=axes[1])
axes[1].set_title("Amount by class (outliers hidden)")
plt.tight_layout()
plt.savefig(FIG_DIR / "01_amount_by_class.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
total_fraud_loss = df.loc[df[TARGET] == 1, "Amount"].sum()
print(f"Total value of fraudulent transactions in the dataset: {total_fraud_loss:,.2f}")
print(f"That is the loss if we do NOTHING - the baseline any model must beat.")

## 4. Time patterns

In [ ]:
tmp = df.copy()
tmp["hour"] = (tmp["Time"] // 3600) % 24     # hour relative to the start of the data

hourly = tmp.groupby("hour")[TARGET].agg(["sum", "count"])
hourly["fraud_rate_pct"] = hourly["sum"] / hourly["count"] * 100

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
hourly["count"].plot(kind="bar", ax=axes[0], color="steelblue")
axes[0].set_title("Transactions per (relative) hour of day")
hourly["fraud_rate_pct"].plot(kind="bar", ax=axes[1], color="firebrick")
axes[1].set_title("Fraud rate (%) per (relative) hour of day")
plt.tight_layout()
plt.savefig(FIG_DIR / "01_hourly_patterns.png", dpi=150, bbox_inches="tight")
plt.show()

`Time` is seconds since the first transaction, so the true clock hour is unknown - this is hour-of-day *relative to the start*.
Volume follows a clear day/night cycle; the fraud *rate* is noisy because there are only ~500 frauds spread over 24 buckets, so I should not over-interpret individual hours.

## 5. Which anonymised features separate fraud from legitimate?

In [ ]:
features = [c for c in df.columns if c.startswith("V")]

# standardised mean difference: how far apart are the class means, in std-dev units?
smd = {}
for f in features:
    fraud, legit = df.loc[df[TARGET] == 1, f], df.loc[df[TARGET] == 0, f]
    pooled_std = np.sqrt((fraud.var() + legit.var()) / 2)
    smd[f] = (fraud.mean() - legit.mean()) / pooled_std
smd = pd.Series(smd).sort_values(key=abs, ascending=False)

fig, ax = plt.subplots(figsize=(8, 6))
smd.head(12)[::-1].plot(kind="barh", color=["firebrick" if v < 0 else "steelblue" for v in smd.head(12)[::-1]], ax=ax)
ax.set_title("Top 12 features by class separation (standardised mean difference)")
ax.set_xlabel("(mean_fraud - mean_legit) / pooled std")
plt.savefig(FIG_DIR / "01_feature_separation.png", dpi=150, bbox_inches="tight")
plt.show()
smd.head(12).round(2)

In [ ]:
top4 = smd.index[:4]
fig, axes = plt.subplots(1, 4, figsize=(16, 3.5))
for ax, f in zip(axes, top4):
    sns.kdeplot(data=df, x=f, hue=TARGET, common_norm=False, fill=True, ax=ax, warn_singular=False)
    ax.set_title(f)
plt.tight_layout()
plt.savefig(FIG_DIR / "01_top_feature_distributions.png", dpi=150, bbox_inches="tight")
plt.show()

**Takeaways**
- A handful of PCA components shift strongly for fraud, so the classes are separable - tree models should do well.
- The features are anonymised, so I can say *which* components matter but not *what* they mean in business terms (a limitation I come back to in `04_insights`).

**My own observations:** _(write 2-3 things you noticed in your plots, e.g. which features separate best, fraud amount vs legitimate amount)_